# Solana Meme Coin Sniper

**Research pipeline:** market discovery → six safety checks → fail-closed final safety gate.

This public notebook intentionally excludes live wallet signing and real-money trading. It demonstrates the screening architecture and produces a research snapshot.

In [ ]:
from pathlib import Path
import os
import sys
import json

# Support running from either the repository root or notebooks/.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT / "src"))

# In a fresh Colab session, install the repository dependencies.
# Skip this line in an already configured local environment if desired.
!pip install -q -r "{REPO_ROOT / 'requirements.txt'}"

print(f"Repository root: {REPO_ROOT}")

In [ ]:
from scanner import SolanaSniper, CONFIG
from checks import SolanaRPC, run_all_checks
from safety_gate import run_final_safety_gate
from execution import preview_trade_candidates

print("✅ Project modules imported")

## 1. Scan current candidates

The scanner uses DEX Screener first, with GeckoTerminal fallbacks. Missing liquidity stays `None` rather than being converted into a false `$0` value.

In [ ]:
scanner = SolanaSniper(CONFIG)
results = scanner.scan_tokens()

print(f"\nCandidates returned: {len(results)}")
scanner.print_results(safe_only=False)

## 2. Configure read-only on-chain checks

Checks 1–6 are read-only. Checks that require Helius use the `HELIUS_API_KEY` environment variable or a Colab Secret with that name. The key is never printed or written to a file.

In [ ]:
def load_colab_or_env_secret(name: str) -> str | None:
    value = os.getenv(name)
    if value:
        return value.strip()

    try:
        from google.colab import userdata
        value = userdata.get(name)
        return value.strip() if value else None
    except Exception:
        return None


HELIUS_API_KEY = load_colab_or_env_secret("HELIUS_API_KEY")

if not HELIUS_API_KEY:
    raise RuntimeError(
        "HELIUS_API_KEY is required for Checks 2, 4, and 6. "
        "Add it to Colab Secrets or the environment and re-run this cell."
    )

HELIUS_RPC_URL = f"https://mainnet.helius-rpc.com/?api-key={HELIUS_API_KEY}"

rpc_client = SolanaRPC(
    rpc_url=HELIUS_RPC_URL,
    request_delay=0.5,
    timeout=20,
    max_retries=3,
)

print("✅ Helius/Solana RPC configured")
print(f"Health: {rpc_client.get_health()}")

## 3. Run Checks 1–6

Each check is read-only and can only remove safety. The final safety decision is made separately by `safety_gate.py`.

In [ ]:
check_outputs = run_all_checks(
    results=results,
    rpc_client=rpc_client,
    helius_api_key=HELIUS_API_KEY,
)

print("✅ Checks 1–6 complete")

## 4. Apply the final safety gate

A token is SAFE only when all six checks are present and pass. Missing or unknown critical information is blocked.

In [ ]:
gate = run_final_safety_gate(
    results=results,
    **check_outputs,
)

print("=" * 90)
print("FINAL SAFETY GATE")
print("=" * 90)
print(f"SAFE:    {gate['safe_count']}")
print(f"BLOCKED: {gate['blocked_count']}")
print(f"TOTAL:   {len(gate['results'])}")

for item in gate["results"]:
    print(f"{item['symbol']:>10} | {item['status']}")

## 5. Non-signing trade preview

The public repository only exposes a preview interface. It identifies tokens that passed the final gate but does not construct, sign, or submit a transaction.

In [ ]:
preview = preview_trade_candidates(results, amount_sol=0.01)
preview

## 6. Save a research snapshot

The snapshot is a local generated artifact and is ignored by Git. It contains scan/check output, not wallet credentials.

In [ ]:
output_path = REPO_ROOT / "results" / "scan_snapshot.json"

payload = {
    "scan_utc": scanner.http.logs[-1]["time_utc"] if scanner.http.logs else None,
    "source_used": scanner.source_used,
    "demo_mode": scanner.demo_mode,
    "gate_completed_at": gate["completed_at"],
    "safe_count": gate["safe_count"],
    "blocked_count": gate["blocked_count"],
    "tokens": [token.to_dict() for token in results],
}

output_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print(f"✅ Saved: {output_path}")

### Important

This project is a research/engineering framework, not a guarantee against scams, rug pulls, or losses. On-chain state and market conditions can change after a scan.

**Live trading remains intentionally disabled in this public version.**